# T6: 2026 R02 中国GP — FP予選シミュレーション vs 実際の予選結果

## 概要
R02中国GPはスプリントウィークエンドのためFP1のみ。  
FP1の予選シミュレーションラップ（CLAUDE.md準拠の個別ラップベース識別）から  
各ドライバーのFP予測ラップタイムを算出し、実際の予選結果と比較する。

### 予選シミュレーション識別ロジック（CLAUDE.md準拠）
1. Softタイヤで記録されたラップ
2. アウトラップ（PitOutTime_secが存在）を除外
3. TyreLife <= 8（新品〜浅い使用状態）
4. セッション最速の103%以内（本気アタックラップ）

## セットアップ & データ読み込み

In [ ]:
import matplotlib
matplotlib.use('Agg')  # GUIなし環境対応

import csv
import os
import math
import statistics
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from scipy import stats

# ===== パス設定 =====
BASE_DIR = '/Volumes/lyssr_workspace/2026_1_4/Motorsports-Visualised'
FP_CSV = os.path.join(BASE_DIR, 'data/2026_R02_China/export/fp_laps.csv')
QUALI_CSV = os.path.join(BASE_DIR, 'data/2026_R02_China/export/quali_laps.csv')
OUTPUT_DIR = os.path.join(BASE_DIR, 'notebooks/output')
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ===== グラフスタイル =====
STYLE = {
    'bg_color': '#1a1a2e',
    'text_color': '#ffffff',
    'grid_color': '#333355',
    'figsize': (12, 6.75),
    'title_size': 18,
    'label_size': 12,
}

# ===== CSV読み込みヘルパー =====
def load_csv(path):
    """CSVをdict listとして読み込む"""
    with open(path, encoding='utf-8') as f:
        return list(csv.DictReader(f))

def to_float(val, default=None):
    """文字列→float変換（空文字・NaN対応）"""
    try:
        v = float(val)
        return v if not math.isnan(v) else default
    except (TypeError, ValueError):
        return default

# FPデータ読み込み
fp_rows = load_csv(FP_CSV)
quali_rows = load_csv(QUALI_CSV)

# セッション種別を動的に確認
sessions_found = sorted(set(r['Session'] for r in fp_rows))
print(f"総FPラップ数: {len(fp_rows)}")
print(f"検出されたセッション: {sessions_found}")
print(f"予選総ラップ数: {len(quali_rows)}")

## 予選シミュレーションラップ特定（CLAUDE.md準拠ロジック）

In [ ]:
def identify_qualisim_laps(rows, session_name=None):
    """
    予選シミュレーションラップを特定する
    1. Softタイヤのみ
    2. アウトラップ除外（PitOutTime_sec存在）
    3. TyreLife <= 8
    4. セッション最速の103%以内
    """
    # セッションフィルタ
    target = [r for r in rows if session_name is None or r['Session'] == session_name]

    # ステップ1: Softタイヤのみ（LapTimeが存在するもの）
    soft_laps = [
        r for r in target
        if r['Compound'] == 'SOFT' and to_float(r['LapTime_sec']) is not None
    ]
    print(f"  [{session_name}] Soft全ラップ: {len(soft_laps)}")

    # ステップ2: アウトラップ除外（PitOutTime_secが空でないもの = アウトラップ）
    no_outlap = [
        r for r in soft_laps
        if not r.get('PitOutTime_sec', '').strip()
    ]
    print(f"  [{session_name}] アウトラップ除外後: {len(no_outlap)}")

    # ステップ3: TyreLife <= 8
    fresh_laps = [
        r for r in no_outlap
        if to_float(r['TyreLife'], 999) <= 8
    ]
    print(f"  [{session_name}] TyreLife<=8後: {len(fresh_laps)}")

    # ステップ4: セッション最速の103%以内
    valid_times = [to_float(r['LapTime_sec']) for r in fresh_laps]
    valid_times = [t for t in valid_times if t is not None]
    if not valid_times:
        return []
    session_best = min(valid_times)
    threshold = session_best * 1.03
    print(f"  [{session_name}] セッションベスト: {session_best:.3f}秒, 103%閾値: {threshold:.3f}秒")

    qualisim = [
        r for r in fresh_laps
        if to_float(r['LapTime_sec'], 9999) <= threshold
    ]
    print(f"  [{session_name}] 最終: {len(qualisim)}ラップ特定")
    return qualisim

# セッション別に予選シミュラップを特定
qualisim_by_session = {}
for sess in sessions_found:
    print(f"\nセッション: {sess}")
    laps = identify_qualisim_laps(fp_rows, sess)
    qualisim_by_session[sess] = laps

# 全セッション統合
all_qualisim = []
for laps in qualisim_by_session.values():
    all_qualisim.extend(laps)
print(f"\n全セッション合計: {len(all_qualisim)}ラップ")

## ドライバー別FP予測ラップタイム

In [ ]:
def get_driver_best_by_session(qualisim_laps):
    """セッション×ドライバーのベストラップタイム辞書を返す"""
    best = {}
    for r in qualisim_laps:
        driver = r['Driver']
        sess = r['Session']
        t = to_float(r['LapTime_sec'])
        if t is None:
            continue
        if driver not in best:
            best[driver] = {}
        if sess not in best[driver] or t < best[driver][sess]:
            best[driver][sess] = t
    return best

def get_fp_prediction(driver_session_best, sessions_available):
    """
    ドライバーごとのFP予測ラップタイムを算出
    FP3優先 -> FP2 -> FP1の優先順
    FP2/FP3が利用可能な場合はトラックエボリューション補正を行う
    """
    SESSION_PRIORITY = ['FP3', 'FP2', 'FP1']
    priority = [s for s in SESSION_PRIORITY if s in sessions_available]

    # トラックエボリューション補正係数（セッション間の中央値差分）
    corrections = {}
    if len(priority) >= 2:
        for i in range(len(priority) - 1):
            sess_a = priority[i]    # より予選に近いセッション
            sess_b = priority[i+1]  # 古いセッション
            diffs = []
            for driver, sbest in driver_session_best.items():
                if sess_a in sbest and sess_b in sbest:
                    diffs.append(sbest[sess_b] - sbest[sess_a])
            if diffs:
                corrections[(sess_b, sess_a)] = statistics.median(diffs)

    predictions = {}
    for driver, sbest in driver_session_best.items():
        # 優先順に最良セッションを選択
        best_time = None
        best_sess = None
        for sess in priority:
            if sess in sbest:
                best_time = sbest[sess]
                best_sess = sess
                break
        if best_time is None:
            continue

        # 古いセッションを使用している場合はトラックエボリューション補正
        corrected = best_time
        if best_sess != priority[0]:
            idx = priority.index(best_sess)
            for k in range(idx):
                older = priority[k + 1]
                newer = priority[k]
                corrected -= corrections.get((older, newer), 0.0)

        predictions[driver] = {
            'fp_time': corrected,
            'fp_raw_time': best_time,
            'fp_session': best_sess,
            'corrected': best_sess != priority[0]
        }
    return predictions

# 実行
driver_session_best = get_driver_best_by_session(all_qualisim)
fp_predictions = get_fp_prediction(driver_session_best, sessions_found)

# FP予測順位リスト表示
print("=== FP予測ラップタイム（ドライバー別）===")
sorted_fp = sorted(fp_predictions.items(), key=lambda x: x[1]['fp_time'])
for rank, (driver, info) in enumerate(sorted_fp, 1):
    mark = " *補正" if info['corrected'] else ""
    print(f"  P{rank:2d}. {driver}: {info['fp_time']:.3f}秒 ({info['fp_session']}{mark})")

print(f"\n注: FP1のみ利用可能（R02はスプリントWEのためFP2/FP3なし）")

## 予選実績 & FP1リザーブドライバー除外

In [ ]:
def get_quali_best(quali_rows):
    """予選ドライバー別ベストラップ（アウトラップ除外）"""
    best = {}
    for r in quali_rows:
        driver = r['Driver']
        t = to_float(r['LapTime_sec'])
        if t is None:
            continue
        # アウトラップ除外
        if r.get('PitOutTime_sec', '').strip():
            continue
        if driver not in best or t < best[driver]:
            best[driver] = t
    return best

# 予選ドライバー一覧（FP1リザーブ除外の基準）
quali_drivers = set(r['Driver'] for r in quali_rows)
quali_best = get_quali_best(quali_rows)

# FP予測を予選出走ドライバーに絞り込む（FP1リザーブ除外）
fp_pred_filtered = {d: info for d, info in fp_predictions.items() if d in quali_drivers}
quali_best_filtered = {d: t for d, t in quali_best.items() if d in fp_pred_filtered}
common_drivers = sorted(set(fp_pred_filtered.keys()) & set(quali_best_filtered.keys()))

# FPデータなし予選ドライバー（FP1不走や識別失敗）
missing_fp = [d for d in quali_drivers if d not in fp_pred_filtered]

print(f"予選ドライバー数: {len(quali_drivers)}")
print(f"FP予測あり予選ドライバー: {len(fp_pred_filtered)}")
print(f"共通分析対象: {len(common_drivers)}名")
if missing_fp:
    print(f"\nFPシミュラップなし（予選出走）: {missing_fp}")
    print("  → これらのドライバーはFP1でシミュラップを行わなかった可能性")

print("\n=== 実際の予選結果（全ドライバー）===")
quali_ranked = sorted(quali_best.items(), key=lambda x: x[1])
for rank, (driver, t) in enumerate(quali_ranked, 1):
    in_analysis = "✓" if driver in common_drivers else "-"
    print(f"  {in_analysis} P{rank:2d}. {driver}: {t:.3f}秒")

## 比較メトリクス計算

In [ ]:
# FP順位と予選順位を算出
fp_times_sorted = sorted([(d, fp_pred_filtered[d]['fp_time']) for d in common_drivers], key=lambda x: x[1])
fp_ranks = {d: rank for rank, (d, _) in enumerate(fp_times_sorted, 1)}
quali_times_sorted = sorted([(d, quali_best_filtered[d]) for d in common_drivers], key=lambda x: x[1])
quali_ranks = {d: rank for rank, (d, _) in enumerate(quali_times_sorted, 1)}

fp_rank_list = [fp_ranks[d] for d in common_drivers]
quali_rank_list = [quali_ranks[d] for d in common_drivers]
fp_time_list = [fp_pred_filtered[d]['fp_time'] for d in common_drivers]
quali_time_list = [quali_best_filtered[d] for d in common_drivers]
time_diffs = [fp_pred_filtered[d]['fp_time'] - quali_best_filtered[d] for d in common_drivers]

# Spearman順位相関
spearman_rho, spearman_p = stats.spearmanr(fp_rank_list, quali_rank_list)
# Pearsonタイム相関
pearson_r, pearson_p = stats.pearsonr(fp_time_list, quali_time_list)
# 順位MAE
rank_errors = [abs(fp_ranks[d] - quali_ranks[d]) for d in common_drivers]
rank_mae = statistics.mean(rank_errors)
# 系統バイアス（FP - Quali の中央値）
systematic_bias = statistics.median(time_diffs)

print("=" * 55)
print("=== 2026 R02 中国GP FP->予選予測 メトリクス ===")
print("=" * 55)
print(f"  Spearman順位相関 (rho): {spearman_rho:.3f}  (p={spearman_p:.4f})")
print(f"  Pearsonタイム相関 (r):  {pearson_r:.3f}  (p={pearson_p:.4f})")
print(f"  順位MAE:               {rank_mae:.2f} ポジション")
print(f"  系統バイアス (中央値):  {systematic_bias:+.3f} 秒")
print()
print(f"  対象ドライバー: {len(common_drivers)}名")
print(f"  使用FPセッション: {sessions_found}")
print()

# 参考: CLAUDE.mdの2025イタリアGP実績
print("【参考: 2025イタリアGP実績（CLAUDE.md）】")
print("  Spearman rho: 0.82")
print("  Pearson r:    0.76")
print("  順位MAE:      2.8 pos")
print("  系統バイアス: +0.34秒")

## ドライバー別予測精度テーブル & CSV出力

In [ ]:
print(f"{'Driver':<6} {'FP順位':>6} {'予選順位':>8} {'順位誤差':>8} {'FP時間':>10} {'予選時間':>10} {'差分':>8} {'セッション':<12}")
print("-" * 78)

table_rows = []
for d in sorted(common_drivers):
    fp_rank = fp_ranks[d]
    q_rank = quali_ranks[d]
    rank_err = fp_rank - q_rank
    fp_t = fp_pred_filtered[d]['fp_time']
    q_t = quali_best_filtered[d]
    diff = fp_t - q_t
    sess = fp_pred_filtered[d]['fp_session']
    corr = "*" if fp_pred_filtered[d]['corrected'] else ""
    print(f"{d:<6} {fp_rank:>6} {q_rank:>8} {rank_err:>+8} {fp_t:>10.3f} {q_t:>10.3f} {diff:>+8.3f} {sess+corr:<12}")
    table_rows.append({
        'Driver': d,
        'FP_Rank': fp_rank,
        'Quali_Rank': q_rank,
        'Rank_Error': rank_err,
        'FP_Time': round(fp_t, 3),
        'Quali_Time': round(q_t, 3),
        'Time_Diff': round(diff, 3),
        'FP_Session': sess + corr
    })

# CSV出力
csv_out = os.path.join(OUTPUT_DIR, 'fp_quali_comparison.csv')
fieldnames = ['Driver', 'FP_Rank', 'Quali_Rank', 'Rank_Error', 'FP_Time', 'Quali_Time', 'Time_Diff', 'FP_Session']
with open(csv_out, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(table_rows)
print(f"\nCSV保存完了: {csv_out}")

## グラフ描画（4パネル比較）

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.patch.set_facecolor(STYLE['bg_color'])
fig.suptitle(
    '2026 R02 China GP - FP Qualisim vs Actual Qualifying',
    color=STYLE['text_color'], fontsize=STYLE['title_size'], fontweight='bold', y=0.98
)

text_c = STYLE['text_color']
grid_c = STYLE['grid_color']
bg_c = STYLE['bg_color']
card_bg = '#1a1a2e'

n = len(common_drivers)
fp_r = [fp_ranks[d] for d in common_drivers]
q_r = [quali_ranks[d] for d in common_drivers]
fp_t_arr = [fp_pred_filtered[d]['fp_time'] for d in common_drivers]
q_t_arr = [quali_best_filtered[d] for d in common_drivers]

# --- Panel 1: FP rank vs Quali rank ---
ax1 = axes[0, 0]
ax1.set_facecolor(card_bg)
ax1.tick_params(colors=text_c)
for spine in ax1.spines.values():
    spine.set_color(grid_c)
ax1.scatter(fp_r, q_r, c='#e10600', s=80, zorder=5, alpha=0.9)
for d in common_drivers:
    ax1.annotate(d, (fp_ranks[d], quali_ranks[d]),
                 textcoords='offset points', xytext=(4, 4), color=text_c, fontsize=7)
diag = list(range(1, n + 1))
ax1.plot(diag, diag, '--', color=grid_c, linewidth=1, alpha=0.7, label='Perfect')
slope, intercept, _, _, _ = stats.linregress(fp_r, q_r)
x_line = np.linspace(1, n, 100)
ax1.plot(x_line, slope * x_line + intercept, '-', color='#FFD700', linewidth=1.5, alpha=0.7, label='Regression')
ax1.set_xlabel('FP Qualisim Rank', color=text_c, fontsize=STYLE['label_size'])
ax1.set_ylabel('Actual Quali Rank', color=text_c, fontsize=STYLE['label_size'])
ax1.set_title(f'Rank Correlation (Spearman rho={spearman_rho:.2f})', color=text_c, fontsize=13)
ax1.grid(True, color=grid_c, alpha=0.4, linestyle='--')
ax1.legend(framealpha=0.3, labelcolor=text_c, facecolor=bg_c, fontsize=8)
ax1.set_xlim(n + 0.5, 0.5)
ax1.set_ylim(n + 0.5, 0.5)

# --- Panel 2: FP time vs Quali time ---
ax2 = axes[0, 1]
ax2.set_facecolor(card_bg)
ax2.tick_params(colors=text_c)
for spine in ax2.spines.values():
    spine.set_color(grid_c)
ax2.scatter(fp_t_arr, q_t_arr, c='#39B54A', s=80, zorder=5, alpha=0.9)
for d in common_drivers:
    ax2.annotate(d, (fp_pred_filtered[d]['fp_time'], quali_best_filtered[d]),
                 textcoords='offset points', xytext=(4, 4), color=text_c, fontsize=7)
min_t = min(min(fp_t_arr), min(q_t_arr)) - 0.5
max_t = max(max(fp_t_arr), max(q_t_arr)) + 0.5
ax2.plot([min_t, max_t], [min_t, max_t], '--', color=grid_c, linewidth=1, alpha=0.7)
ax2.plot([min_t, max_t], [min_t - systematic_bias, max_t - systematic_bias],
         '-', color='#FF9900', linewidth=1.5, alpha=0.7,
         label=f'Bias {systematic_bias:+.3f}s')
ax2.set_xlabel('FP Qualisim Time (s)', color=text_c, fontsize=STYLE['label_size'])
ax2.set_ylabel('Actual Quali Time (s)', color=text_c, fontsize=STYLE['label_size'])
ax2.set_title(f'Time Correlation (Pearson r={pearson_r:.2f})', color=text_c, fontsize=13)
ax2.grid(True, color=grid_c, alpha=0.4, linestyle='--')
ax2.legend(framealpha=0.3, labelcolor=text_c, facecolor=bg_c, fontsize=8)

# --- Panel 3: Rank error per driver ---
ax3 = axes[1, 0]
ax3.set_facecolor(card_bg)
ax3.tick_params(colors=text_c)
for spine in ax3.spines.values():
    spine.set_color(grid_c)
sorted_by_q = sorted(common_drivers, key=lambda d: quali_ranks[d])
rank_errs = [fp_ranks[d] - quali_ranks[d] for d in sorted_by_q]
bar_colors = ['#e10600' if e < 0 else '#39B54A' for e in rank_errs]
ax3.bar(range(len(sorted_by_q)), rank_errs, color=bar_colors, alpha=0.85, zorder=5)
ax3.set_xticks(range(len(sorted_by_q)))
ax3.set_xticklabels(sorted_by_q, rotation=45, ha='right', color=text_c, fontsize=8)
ax3.axhline(0, color=text_c, linewidth=0.8, alpha=0.5)
ax3.set_xlabel('Driver (by Quali rank)', color=text_c, fontsize=STYLE['label_size'])
ax3.set_ylabel('Rank Error (FP - Actual)', color=text_c, fontsize=STYLE['label_size'])
ax3.set_title(f'Per-Driver Rank Error (MAE={rank_mae:.2f} pos)', color=text_c, fontsize=13)
ax3.grid(True, color=grid_c, alpha=0.4, linestyle='--', axis='y')
red_p = mpatches.Patch(color='#e10600', alpha=0.85, label='FP over-estimated')
green_p = mpatches.Patch(color='#39B54A', alpha=0.85, label='FP under-estimated')
ax3.legend(handles=[red_p, green_p], framealpha=0.3, labelcolor=text_c, facecolor=bg_c, fontsize=7)

# --- Panel 4: Time diff histogram + metrics summary ---
ax4 = axes[1, 1]
ax4.set_facecolor(card_bg)
ax4.tick_params(colors=text_c)
for spine in ax4.spines.values():
    spine.set_color(grid_c)
ax4.hist(time_diffs, bins=8, color='#0072CE', alpha=0.8, edgecolor=grid_c, zorder=5)
ax4.axvline(0, color=text_c, linewidth=1, alpha=0.6, linestyle='--', label='Zero diff')
ax4.axvline(systematic_bias, color='#FF9900', linewidth=1.5, alpha=0.8,
            label=f'Median {systematic_bias:+.3f}s')
ax4.set_xlabel('FP Time - Quali Time (s)', color=text_c, fontsize=STYLE['label_size'])
ax4.set_ylabel('Drivers', color=text_c, fontsize=STYLE['label_size'])
ax4.set_title('Time Diff Distribution', color=text_c, fontsize=13)
ax4.grid(True, color=grid_c, alpha=0.4, linestyle='--', axis='y')
ax4.legend(framealpha=0.3, labelcolor=text_c, facecolor=bg_c, fontsize=8)
metrics_text = (
    f"Spearman rho: {spearman_rho:.3f}\n"
    f"Pearson r:    {pearson_r:.3f}\n"
    f"Rank MAE:     {rank_mae:.2f} pos\n"
    f"Bias (med):   {systematic_bias:+.3f}s\n"
    f"N drivers:    {len(common_drivers)}\n"
    f"FP sessions:  {', '.join(sessions_found)}"
)
ax4.text(0.97, 0.97, metrics_text,
         transform=ax4.transAxes, va='top', ha='right',
         color=text_c, fontsize=9,
         bbox=dict(boxstyle='round', facecolor='#1c1c25', alpha=0.8, edgecolor=grid_c))

plt.tight_layout(rect=[0, 0, 1, 0.96])
png_out = os.path.join(OUTPUT_DIR, 'fp_quali_scatter.png')
plt.savefig(png_out, dpi=150, bbox_inches='tight', facecolor=STYLE['bg_color'])
plt.show()
print(f"PNG saved: {png_out}")